# Clean WR Prediction Pipeline

This notebook is the cleaned-up version of the original exploration notebook. The goal is to keep the current working data pipeline in one readable place: load data, calculate PPR, create prediction-safe features, evaluate baselines, and train the first Linear Regression model.

## 1. Imports

Import the libraries used in the notebook. The main dataframe library is Polars; scikit-learn is introduced later for Linear Regression.

In [36]:
import pandas as pd
import nflreadpy as nfl
import polars as pl
import matplotlib.pyplot as plt

## 2. Load Weekly Player Data

Load one season of weekly NFL player stats from `nflreadpy`. At this stage, the data still includes all positions and may include postseason rows.

In [37]:
weekly = nfl.load_player_stats([2025])
weekly

player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,…,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""00-0023459""","""A.Rodgers""","""Aaron Rodgers""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2025,1,"""REG""","""2025_01_PIT_NYJ""","""PIT""","""NYJ""",22,30,244,4,0,4,-26,0,0,139,173,14,10.204755,6.350381,0,1.755396,11,8,5,0,1,-1,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,25.66,25.66
"""00-0023853""","""M.Prater""","""Matt Prater""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2025,1,"""REG""","""2025_01_BAL_BUF""","""BUF""","""BAL""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,"""25;43;32""",null,null,100,0,0,2,2,0,0,1.0,1,1,0,0,32,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0025565""","""N.Folk""","""Nick Folk""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2025,1,"""REG""","""2025_01_PIT_NYJ""","""NYJ""","""PIT""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,"""35;51""",null,null,86,0,0,2,2,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0026158""","""J.Flacco""","""Joe Flacco""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2025,1,"""REG""","""2025_01_CIN_CLE""","""CLE""","""CIN""",31,45,290,1,2,2,-12,0,0,273,173,13,-1.842128,2.374263,0,1.062271,11,6,3,0,2,6,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,12.2,12.2
"""00-0026190""","""C.Campbell""","""Calais Campbell""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2025,1,"""REG""","""2025_01_ARI_NO""","""ARI""","""NO""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00-0040699""","""W.Campbell""","""Will Campbell""","""OT""","""OL""","""https://static.www.nfl.com/ima…",2025,22,"""POST""","""2025_22_SEA_NE""","""NE""","""SEA""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0040716""","""C.Woodson""","""Craig Woodson""","""SAF""","""DB""","""https://static.www.nfl.com/ima…",2025,22,"""POST""","""2025_22_SEA_NE""","""NE""","""SEA""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0040733""","""N.Emmanwori""","""Nick Emmanwori""","""SAF""","""DB""","""https://static.www.nfl.com/ima…",2025,22,"""POST""","""2025_22_SEA_NE""","""SEA""","""NE""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,

## 3. Filter To Wide Receivers And Select Useful Columns

Keep only WR rows and select the columns needed for scoring, feature engineering, baselines, and modeling. A future script should explicitly filter `season_type == "REG"` instead of relying on week numbers later.

In [38]:
receivers = weekly.filter(weekly['position'] == 'WR')
receiving_stats = receivers.select([
    "player_display_name",
    "season",
    "week",
    "team",
    "opponent_team",
    "receptions",
    "targets",
    "receiving_yards",
    "receiving_tds",
    "receiving_fumbles_lost",
    "receiving_2pt_conversions",
    "special_teams_tds",
    "rushing_yards",
    "rushing_tds",
    "rushing_fumbles_lost",
    "rushing_2pt_conversions",
    "fantasy_points_ppr",
])

receiving_stats

player_display_name,season,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr
str,i32,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64
"""Adam Thielen""",2025,1,"""MIN""","""CHI""",0,1,0,0,0,1,0,0,0,0,0,2.0
"""Keenan Allen""",2025,1,"""LAC""","""KC""",7,10,68,1,0,0,0,0,0,0,0,19.8
"""DeAndre Hopkins""",2025,1,"""BAL""","""BUF""",2,2,35,1,0,0,0,0,0,0,0,11.5
"""Brandin Cooks""",2025,1,"""NO""","""ARI""",3,4,26,0,0,0,0,0,0,0,0,5.6
"""Davante Adams""",2025,1,"""LA""","""HOU""",4,8,51,0,0,0,0,0,0,0,0,9.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jaxon Smith-Njigba""",2025,22,"""SEA""","""NE""",4,10,27,0,0,0,0,0,0,0,0,6.7
"""Kayshon Boutte""",2025,22,"""NE""","""SEA""",1,5,21,0,0,0,0,0,0,0,0,3.1
"""DeMario Douglas""",2025,22,"""NE""","""SEA""",5,7,45,0,0,0,0,0,0,0,0,9.5


## 4. Calculate Custom Full-PPR Scoring

Create `calculated_ppr` from the MVP scoring rules: receptions, receiving/rushing yards, receiving/rushing touchdowns, and receiving/rushing fumbles lost.

In [39]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.scoring import calculate_ppr

receiving_stats_ppr = calculate_ppr(receiving_stats)
receiving_stats_ppr

player_display_name,season,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr,calculated_ppr
str,i32,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""Adam Thielen""",2025,1,"""MIN""","""CHI""",0,1,0,0,0,1,0,0,0,0,0,2.0,0.0
"""Keenan Allen""",2025,1,"""LAC""","""KC""",7,10,68,1,0,0,0,0,0,0,0,19.8,19.8
"""DeAndre Hopkins""",2025,1,"""BAL""","""BUF""",2,2,35,1,0,0,0,0,0,0,0,11.5,11.5
"""Brandin Cooks""",2025,1,"""NO""","""ARI""",3,4,26,0,0,0,0,0,0,0,0,5.6,5.6
"""Davante Adams""",2025,1,"""LA""","""HOU""",4,8,51,0,0,0,0,0,0,0,0,9.1,9.1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jaxon Smith-Njigba""",2025,22,"""SEA""","""NE""",4,10,27,0,0,0,0,0,0,0,0,6.7,6.7
"""Kayshon Boutte""",2025,22,"""NE""","""SEA""",1,5,21,0,0,0,0,0,0,0,0,3.1,3.1
"""DeMario Douglas""",2025,22,"""NE""","""SEA""",5,7,45,0,0,0,0,0,0,0,0,9.5,9.5


## 5. Basic Season Averages

Calculate simple player-level averages to summarize season production. These are useful for exploration, but prediction features must be shifted so they only use past games.

In [40]:
player_averages = receiving_stats_ppr.group_by('player_display_name').agg([
    pl.col("calculated_ppr").mean().round(2).alias("avg_calculated_ppr"),
    pl.col("receptions").mean().round(2).alias("avg_receptions"),
    pl.col("receiving_yards").mean().round(2).alias("avg_receiving_yards"),
]).sort("avg_calculated_ppr", descending=True)

player_averages

player_display_name,avg_calculated_ppr,avg_receptions,avg_receiving_yards
str,f64,f64,f64
"""Puka Nacua""",23.82,8.05,107.74
"""Jaxon Smith-Njigba""",20.44,6.8,99.6
"""Ja'Marr Chase""",19.6,7.81,88.25
"""Amon-Ra St. Brown""",19.06,6.88,82.41
"""Rashee Rice""",18.51,6.62,71.38
…,…,…,…
"""Kaden Davis""",0.0,0.0,0.0
"""Jamal Agnew""",0.0,0.0,0.0
"""Jeshaun Jones""",0.0,0.0,0.0


## 6. Previous 3-Game Features

Create rolling 3-game averages that are safe for prediction. The `shift(1)` step is critical: it ensures each row uses only games before the week being predicted.

In [41]:
player_prediction_features = (
    receiving_stats_ppr
    .sort(["player_display_name", "week"])
    .with_columns(
        pl.col("calculated_ppr")
        .shift(1)
        .rolling_mean(window_size=3, min_samples=3)
        .round(2)
        .over("player_display_name")
        .alias("prev_rolling_3_ppr"),

        pl.col("targets")
        .shift(1)
        .rolling_mean(window_size=3, min_samples=3)
        .round(2)
        .over("player_display_name")
        .alias("prev_rolling_3_targets"),

        pl.col("receptions")
        .shift(1)
        .rolling_mean(window_size=3, min_samples=3)
        .round(2)
        .over("player_display_name")
        .alias("prev_rolling_3_receptions"),

        pl.col("receiving_yards")
        .shift(1)
        .rolling_mean(window_size=3, min_samples=3)
        .round(2)
        .over("player_display_name")
        .alias("prev_rolling_3_receiving_yards")
    )
)

## 7. Season-To-Date Features

Create season-to-date averages using only previous games. These features represent a player's overall prior-season level before each week.

In [42]:
player_prediction_features = (
    player_prediction_features
    .sort(["player_display_name", "week"])
    .with_columns(
        (
            pl.col("calculated_ppr").shift(1).cum_sum().over("player_display_name")
            /
            pl.col("calculated_ppr").shift(1).cum_count().over("player_display_name")
        )
        .round(2)
        .alias("prev_season_avg_ppr"),
    )
)

## 8. Baseline Predictions

Create simple baseline predictions from previous 3-game PPR and season-to-date PPR. These are the benchmarks the ML model must beat.

In [43]:
baseline_predictions = (
    player_prediction_features
    .with_columns(
        pl.col("prev_rolling_3_ppr").alias("baseline_prediction"),
        pl.col("prev_season_avg_ppr").alias("baseline_prediction_season")
    )
)

baseline_predictions.filter(pl.col('player_display_name') == 'Jaxon Smith-Njigba').select([
    'player_display_name',
    'week',
    'calculated_ppr',
    'baseline_prediction',
    'baseline_prediction_season',
    'prev_rolling_3_ppr',
    'prev_rolling_3_receptions',
    'prev_rolling_3_receiving_yards',
])

player_display_name,week,calculated_ppr,baseline_prediction,baseline_prediction_season,prev_rolling_3_ppr,prev_rolling_3_receptions,prev_rolling_3_receiving_yards
str,i32,f64,f64,f64,f64,f64,f64
"""Jaxon Smith-Njigba""",1,19.4,null,null,null,null,null
"""Jaxon Smith-Njigba""",2,18.3,null,19.4,null,null,null
"""Jaxon Smith-Njigba""",3,20.6,null,18.85,null,null,null
"""Jaxon Smith-Njigba""",4,13.0,19.43,19.43,19.43,7.33,107.67
"""Jaxon Smith-Njigba""",5,27.2,17.3,17.83,17.3,5.67,92.67
…,…,…,…,…,…,…,…
"""Jaxon Smith-Njigba""",17,16.2,23.33,21.95,23.33,7.33,100.33
"""Jaxon Smith-Njigba""",18,14.4,19.37,21.59,19.37,8.0,93.67
"""Jaxon Smith-Njigba""",20,10.9,18.07,21.17,18.07,7.67,84.0


## 9. Baseline Errors

Filter to valid baseline rows and calculate absolute/signed errors. The notebook currently excludes postseason rows with `week <= 18`; this should become an explicit regular-season filter in reusable code.

In [ ]:
baseline_prediction_clean = baseline_predictions.filter(
    (pl.col("prev_rolling_3_ppr").is_not_null()) & (pl.col("week") <= 18)
)

baseline_prediction_clean = baseline_prediction_clean.with_columns(
    (pl.col("baseline_prediction") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("absolute_error"),

    (pl.col("calculated_ppr") - pl.col("baseline_prediction"))
    .round(2)
    .alias("signed_error"),

    (pl.col("calculated_ppr") - pl.col("prev_season_avg_ppr"))
    .abs()
    .round(2)
    .alias("season_avg_error"),

    (pl.col("calculated_ppr") - pl.col("prev_season_avg_ppr"))
    .round(2)
    .alias("signed_season_avg_error")
)


## 10. Previous 3-Game Baseline MAE

Measure previous 3-game baseline performance on the same Weeks 15-18 test window used for Linear Regression, so comparisons are fair.

In [45]:
mean_abs_error = baseline_prediction_clean.filter((pl.col('week') >= 15) & (pl.col('week') <= 18)).select(
    pl.col("absolute_error").mean().round(2).alias("mean_absolute_error")
)
mean_abs_error

mean_absolute_error
f64
4.47


## 11. Season-To-Date Baseline MAE

Measure the season-to-date baseline on the same Weeks 15-18 test window.

In [ ]:
mean_season_avg_error = baseline_prediction_clean.filter((pl.col('week') >= 15) & (pl.col('week') <= 18)).select(
    pl.col("season_avg_error").mean().round(2).alias("mean_season_avg_error")
)
mean_season_avg_error

mean_season_avg_error
f64
4.27


## 12. ML-Ready Dataset

Build the modeling table. Each row is one WR-week. Feature columns contain information available before the game; `calculated_ppr` is the target to predict.

In [47]:
dataset = (
    baseline_prediction_clean.select([
        'player_display_name',
        'season',
        'week',
        'team',
        'opponent_team',
        'prev_rolling_3_ppr',
        'prev_rolling_3_targets',
        'prev_rolling_3_receptions',
        'prev_rolling_3_receiving_yards',
        'prev_season_avg_ppr',
        'calculated_ppr',
    ])
)

dataset = dataset.filter(
    (pl.col("prev_rolling_3_ppr").is_not_null()) & 
    (pl.col("prev_rolling_3_targets").is_not_null()) & 
    (pl.col("prev_rolling_3_receptions").is_not_null()) & 
    (pl.col("prev_rolling_3_receiving_yards").is_not_null()) & 
    (pl.col("prev_season_avg_ppr").is_not_null()) & 
    (pl.col("calculated_ppr").is_not_null())
)

dataset

player_display_name,season,week,team,opponent_team,prev_rolling_3_ppr,prev_rolling_3_targets,prev_rolling_3_receptions,prev_rolling_3_receiving_yards,prev_season_avg_ppr,calculated_ppr
str,i32,i32,str,str,f64,f64,f64,f64,f64,f64
"""A.J. Brown""",2025,4,"""PHI""","""TB""",10.8,6.33,4.0,48.0,10.8,2.7
"""A.J. Brown""",2025,5,"""PHI""","""DEN""",11.1,9.0,4.33,47.67,8.78,9.3
"""A.J. Brown""",2025,6,"""PHI""","""NYG""",11.63,9.0,4.33,53.0,8.88,14.0
"""A.J. Brown""",2025,7,"""PHI""","""MIN""",8.67,8.67,4.33,43.33,9.73,28.1
"""A.J. Brown""",2025,10,"""PHI""","""GB""",17.13,7.67,5.0,81.33,12.36,3.3
…,…,…,…,…,…,…,…,…,…,…
"""Zay Flowers""",2025,17,"""BAL""","""GB""",19.2,7.67,6.0,92.0,13.37,13.0
"""Zay Flowers""",2025,18,"""BAL""","""PIT""",16.67,5.67,4.67,60.67,13.34,29.8
"""Zay Jones""",2025,6,"""ARI""","""IND""",2.9,2.33,1.67,12.33,2.9,12.9


## 13. Linear Regression Model

Train the first simple ML model using a temporal split: earlier weeks for training, later weeks for testing. This avoids random-split leakage.

In [48]:
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


X_train = dataset.filter((pl.col('week') >= 4) & (pl.col('week') <= 14)).select([
    'prev_rolling_3_ppr',
    'prev_rolling_3_targets',
    'prev_rolling_3_receptions',
    'prev_rolling_3_receiving_yards',
    'prev_season_avg_ppr',
]).to_numpy()

X_test = dataset.filter((pl.col('week') >= 15) & (pl.col('week') <= 18)).select([
    'prev_rolling_3_ppr',
    'prev_rolling_3_targets',
    'prev_rolling_3_receptions',
    'prev_rolling_3_receiving_yards',
    'prev_season_avg_ppr',
]).to_numpy()

y_train = dataset.filter((pl.col('week') >= 4) & (pl.col('week') <= 14)).select('calculated_ppr').to_numpy().ravel()
y_test = dataset.filter((pl.col('week') >= 15) & (pl.col('week') <= 18)).select('calculated_ppr').to_numpy().ravel()

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae}")


Mean Absolute Error: 4.2328067779215965


## 14. Error Inspection

Create a test-results table to inspect where Linear Regression misses badly and compare those misses against the simple baselines.

In [49]:
test_results = (
    dataset
    .filter((pl.col('week') >= 15) & (pl.col('week') <= 18))
    .with_columns(
        pl.Series("linear_regression_prediction", y_pred)
    )
)

test_results = test_results.with_columns(
    (pl.col("linear_regression_prediction") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("linear_regression_absolute_error"),

    (pl.col("prev_rolling_3_ppr") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("rolling_baseline_absolute_error"),

    (pl.col("prev_season_avg_ppr") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("season_baseline_absolute_error")
)

test_results.select([
    "player_display_name",
    "week",
    "calculated_ppr",
    "linear_regression_prediction",
    "linear_regression_absolute_error",
    "rolling_baseline_absolute_error",
    "season_baseline_absolute_error",
]).sort("linear_regression_absolute_error", descending=True).head(20)

player_display_name,week,calculated_ppr,linear_regression_prediction,linear_regression_absolute_error,rolling_baseline_absolute_error,season_baseline_absolute_error
str,i32,f64,f64,f64,f64,f64
"""Puka Nacua""",16,46.5,17.935538,28.56,20.9,24.44
"""Amon-Ra St. Brown""",15,41.4,14.346647,27.05,26.37,23.44
"""Chris Olave""",16,36.8,12.413326,24.39,23.07,21.99
"""Alec Pierce""",18,29.2,7.878945,21.32,20.13,18.19
"""Luther Burden III""",17,27.8,8.938561,18.86,16.8,20.68
…,…,…,…,…,…,…
"""CeeDee Lamb""",18,1.4,14.875511,13.48,11.2,15.22
"""Drake London""",17,1.4,14.833636,13.43,14.27,16.67
"""Kalif Raymond""",16,16.2,3.165811,13.03,13.3,13.29


## 15. Model Coefficients

Inspect Linear Regression coefficients to understand which features the model leans on. Coefficients can be tricky when features overlap, but they are still useful for interpretation.

In [50]:
feature_columns = [
    "prev_rolling_3_ppr",
    "prev_rolling_3_targets",
    "prev_rolling_3_receptions",
    "prev_rolling_3_receiving_yards",
    "prev_season_avg_ppr",
]

coefficients = pl.DataFrame({
    "feature": feature_columns,
    "coefficient": model.coef_,
}).sort("coefficient", descending=True)

coefficients

feature,coefficient
str,f64
"""prev_rolling_3_receptions""",0.551627
"""prev_rolling_3_targets""",0.452256
"""prev_season_avg_ppr""",0.451323
"""prev_rolling_3_receiving_yards""",-0.003619
"""prev_rolling_3_ppr""",-0.050509
